In [1]:

import re
import html
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:

PROJECT_DIR = Path(
    r"C:\Users\bimal\OneDrive\Desktop\satya\ML Projects"
    r"\question_similarity"
)

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

TRAIN_PATH = RAW_DATA_DIR / "train.csv"
TEST_PATH = RAW_DATA_DIR / "test.csv"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = "is_duplicate"
QUESTION_COLUMNS = ["question1", "question2"]

print("Project directory:", PROJECT_DIR)
print("Training data:", TRAIN_PATH)
print("Processed data:", PROCESSED_DATA_DIR)

Project directory: C:\Users\bimal\OneDrive\Desktop\satya\ML Projects\question_similarity
Training data: C:\Users\bimal\OneDrive\Desktop\satya\ML Projects\question_similarity\data\raw\train.csv
Processed data: C:\Users\bimal\OneDrive\Desktop\satya\ML Projects\question_similarity\data\processed


In [3]:

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

display(train_df.head())
display(test_df.head())

C:\Users\bimal\AppData\Local\Temp\ipykernel_35628\3563661465.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv(TEST_PATH)


Training shape: (404290, 6)
Testing shape: (3563475, 3)


,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


,test_id,question1,question2
0,0,How does the Surface Pro himself 4 compare wit...,Why did Microsoft choose core m3 and not core ...
1,1,Should I have a hair transplant at age 24? How...,How much cost does hair transplant require?
2,2,What but is the best way to send money from Ch...,What you send money to China?
3,3,Which food not emulsifiers?,What foods fibre?
4,4,"How ""aberystwyth"" start reading?",How their can I start reading?


In [4]:

required_train_columns = {
    "question1",
    "question2",
    "is_duplicate"
}

required_test_columns = {
    "question1",
    "question2"
}

assert required_train_columns.issubset(train_df.columns), (
    f"Missing training columns: "
    f"{required_train_columns - set(train_df.columns)}"
)

assert required_test_columns.issubset(test_df.columns), (
    f"Missing test columns: "
    f"{required_test_columns - set(test_df.columns)}"
)

assert set(train_df[TARGET_COLUMN].dropna().unique()).issubset({0, 1}), (
    "Target must contain only binary values 0 and 1."
)

print("Dataset validation completed successfully.")

Dataset validation completed successfully.


In [5]:
# Handle missing question values

for df in (train_df, test_df):
    for col in ["question1", "question2"]:
        df[col] = df[col].fillna("").astype(str)

print("Missing question values handled successfully.")

Missing question values handled successfully.


In [6]:
def normalize_text(text):
    if not isinstance(text, str):
        return ""

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Decode HTML entities
    text = html.unescape(text)

    # Convert to lowercase
    text = text.lower()

    # Replace multiple whitespace characters with one space
    text = re.sub(r"\s+", " ", text)

    # Remove leading and trailing spaces
    text = text.strip()

    return text


# Apply normalization to both question columns
for col in ["question1", "question2"]:
    train_df[col] = train_df[col].apply(normalize_text)
    test_df[col] = test_df[col].apply(normalize_text)

print("Text normalization completed.")

Text normalization completed.


In [7]:
contractions = {
    "can't": "cannot",
    "won't": "will not",
    "n't": " not",
    "it's": "it is",
    "i'm": "i am",
    "you're": "you are",
    "we're": "we are",
    "they're": "they are",
    "he's": "he is",
    "she's": "she is",
    "that's": "that is",
    "what's": "what is",
    "who's": "who is",
    "where's": "where is",
    "there's": "there is",
    "i've": "i have",
    "we've": "we have",
    "they've": "they have",
    "i'll": "i will",
    "you'll": "you will",
    "we'll": "we will",
    "they'll": "they will",
    "i'd": "i would",
    "you'd": "you would",
    "we'd": "we would",
    "they'd": "they would"
}

# Match complete contractions, longest first
contraction_pattern = re.compile(
    r"\b(?:" + "|".join(
        re.escape(key) for key in sorted(
            contractions, key=len, reverse=True
        )
    ) + r")\b",
    flags=re.IGNORECASE
)

def expand_contractions(text):
    return contraction_pattern.sub(
        lambda match: contractions[match.group().lower()],
        text
    )


for col in ["question1", "question2"]:
    train_df[col] = train_df[col].apply(expand_contractions)
    test_df[col] = test_df[col].apply(expand_contractions)

print("Contraction expansion completed.")

Contraction expansion completed.


In [8]:

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove non-printable control characters
    text = "".join(
        char for char in text
        if char.isprintable() or char.isspace()
    )

    # Replace multiple whitespace characters with one space
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# Apply cleaning to both question columns
for col in ["question1", "question2"]:
    train_df[col] = train_df[col].apply(clean_text)
    test_df[col] = test_df[col].apply(clean_text)

print("Unnecessary character cleaning completed.")

Unnecessary character cleaning completed.


In [9]:


import re

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove non-printable control characters
    text = "".join(
        char for char in text
        if char.isprintable() or char.isspace()
    )

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


for col in ["question1", "question2"]:
    train_df[col] = train_df[col].apply(clean_text)
    test_df[col] = test_df[col].apply(clean_text)

print("Final text cleaning completed.")

Final text cleaning completed.


In [10]:

# Check for missing values
assert train_df[["question1", "question2"]].isna().sum().sum() == 0
assert test_df[["question1", "question2"]].isna().sum().sum() == 0

# Check data types
assert train_df[["question1", "question2"]].dtypes.eq("object").all()
assert test_df[["question1", "question2"]].dtypes.eq("object").all()

# Check row counts
assert len(train_df) == len(train_df.index)
assert len(test_df) == len(test_df.index)

print("Preprocessing validation completed successfully.")

Preprocessing validation completed successfully.


In [11]:

from pathlib import Path

PROCESSED_DIR = Path(
    r"C:\Users\bimal\OneDrive\Desktop\satya\ML Projects"
    r"\question_similarity\data\processed"
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_OUTPUT = PROCESSED_DIR / "train_preprocessed.csv"
TEST_OUTPUT = PROCESSED_DIR / "test_preprocessed.csv"

train_df.to_csv(TRAIN_OUTPUT, index=False)
test_df.to_csv(TEST_OUTPUT, index=False)

print("Preprocessed datasets saved successfully.")
print("Training:", TRAIN_OUTPUT)
print("Testing:", TEST_OUTPUT)

Preprocessed datasets saved successfully.
Training: C:\Users\bimal\OneDrive\Desktop\satya\ML Projects\question_similarity\data\processed\train_preprocessed.csv
Testing: C:\Users\bimal\OneDrive\Desktop\satya\ML Projects\question_similarity\data\processed\test_preprocessed.csv


In [12]:

# Reload saved files to verify successful storage
train_processed = pd.read_csv(TRAIN_OUTPUT)
test_processed = pd.read_csv(TEST_OUTPUT)

print("Training dataset:", train_processed.shape)
print("Testing dataset:", test_processed.shape)

print("\nTraining sample:")
display(train_processed[
    ["question1", "question2", "is_duplicate"]
].head())

print("\nTesting sample:")
display(test_processed[
    ["question1", "question2"]
].head())


C:\Users\bimal\AppData\Local\Temp\ipykernel_35628\1654456883.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  test_processed = pd.read_csv(TEST_OUTPUT)


Training dataset: (404290, 6)
Testing dataset: (3563475, 3)

Training sample:


,question1,question2,is_duplicate
0,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,0
1,what is the story of kohinoor (koh-i-noor) dia...,what would happen if the indian government sto...,0
2,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,0
3,why am i mentally very lonely? how can i solve...,find the remainder when [math]23^{24}[/math] i...,0
4,"which one dissolve in water quikly sugar, salt...",which fish would survive in salt water?,0



Testing sample:


,question1,question2
0,how does the surface pro himself 4 compare wit...,why did microsoft choose core m3 and not core ...
1,should i have a hair transplant at age 24? how...,how much cost does hair transplant require?
2,what but is the best way to send money from ch...,what you send money to china?
3,which food not emulsifiers?,what foods fibre?
4,"how ""aberystwyth"" start reading?",how their can i start reading?
